In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("PHASE 2: FEATURE ENGINEERING")
print("="*60)

PHASE 2: FEATURE ENGINEERING


In [2]:
# Load dataset
print("\n1. LOADING DATA...")
df = pd.read_csv('data/European_Bank.csv')
print(f"   Original shape: {df.shape}")
df.head()


1. LOADING DATA...
   Original shape: (10000, 14)


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Remove non-informative columns
print("\n2. REMOVING NON-INFORMATIVE COLUMNS...")
df = df.drop(['CustomerId', 'Surname'], axis=1)
print(f"   Removed: CustomerId, Surname")
print(f"   New shape: {df.shape}")
df.head()


2. REMOVING NON-INFORMATIVE COLUMNS...
   Removed: CustomerId, Surname
   New shape: (10000, 12)


,Year,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Create derived features
print("\n3. CREATING DERIVED FEATURES...")

# Balance to Salary Ratio
df['Balance_to_Salary'] = df['Balance'] / (df['EstimatedSalary'] + 1)
df['Balance_to_Salary'] = df['Balance_to_Salary'].clip(0, 5)

# Age and Tenure interaction
df['Age_Tenure'] = df['Age'] * df['Tenure']

# Product Density
df['Product_Density'] = df['NumOfProducts'] / (df['Tenure'] + 1)

# Product Engagement Score
df['Product_Engagement'] = df['NumOfProducts'] * df['IsActiveMember']

# Wealth Index
df['Wealth_Index'] = df['Balance'] + (df['EstimatedSalary'] / 12)

# Account Utilization
df['Account_Utilization'] = df['Balance'] / (df['NumOfProducts'] + 1)

# High Value Customer flag
df['High_Value'] = ((df['Balance'] > 50000) | (df['EstimatedSalary'] > 100000)).astype(int)

# Credit Score Category
df['Credit_Category'] = pd.cut(df['CreditScore'], 
                                bins=[0, 580, 670, 740, 850], 
                                labels=['Poor', 'Fair', 'Good', 'Excellent'])

print(f"   Added features:")
print(f"      - Balance_to_Salary")
print(f"      - Age_Tenure")
print(f"      - Product_Density")
print(f"      - Product_Engagement")
print(f"      - Wealth_Index")
print(f"      - Account_Utilization")
print(f"      - High_Value")
print(f"      - Credit_Category")


3. CREATING DERIVED FEATURES...
   Added features:
      - Balance_to_Salary
      - Age_Tenure
      - Product_Density
      - Product_Engagement
      - Wealth_Index
      - Account_Utilization
      - High_Value
      - Credit_Category


In [5]:
# Encode categorical variables
print("\n4. ENCODING CATEGORICAL VARIABLES...")

# One-hot encoding for Geography and Gender
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)
print(f"   One-hot encoded: Geography_Germany, Geography_Spain, Gender_Male")

# Label encoding for Credit_Category
le = LabelEncoder()
df['Credit_Category'] = le.fit_transform(df['Credit_Category'])
print(f"   Label encoded: Credit_Category (Poor=0, Fair=1, Good=2, Excellent=3)")

print(f"\n   Final columns: {list(df.columns)}")


4. ENCODING CATEGORICAL VARIABLES...
   One-hot encoded: Geography_Germany, Geography_Spain, Gender_Male
   Label encoded: Credit_Category (Poor=0, Fair=1, Good=2, Excellent=3)

   Final columns: ['Year', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Balance_to_Salary', 'Age_Tenure', 'Product_Density', 'Product_Engagement', 'Wealth_Index', 'Account_Utilization', 'High_Value', 'Credit_Category', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']


In [6]:
# Separate features and target
print("\n5. PREPARING FEATURES AND TARGET...")
X = df.drop('Exited', axis=1)
y = df['Exited']

print(f"   Features shape: {X.shape}")
print(f"   Features: {list(X.columns)}")
print(f"   Target shape: {y.shape}")
print(f"   Target distribution:")
print(f"      Retained: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"      Churned:  {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")


5. PREPARING FEATURES AND TARGET...
   Features shape: (10000, 20)
   Features: ['Year', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Balance_to_Salary', 'Age_Tenure', 'Product_Density', 'Product_Engagement', 'Wealth_Index', 'Account_Utilization', 'High_Value', 'Credit_Category', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']
   Target shape: (10000,)
   Target distribution:
      Retained: 7963 (79.6%)
      Churned:  2037 (20.4%)


In [7]:
# Scale numerical features
print("\n6. SCALING NUMERICAL FEATURES...")
numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 
                  'EstimatedSalary', 'Balance_to_Salary', 'Age_Tenure',
                  'Product_Density', 'Product_Engagement', 'Wealth_Index',
                  'Account_Utilization']

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
print(f"   Scaled {len(numerical_cols)} numerical features")


6. SCALING NUMERICAL FEATURES...
   Scaled 12 numerical features


In [8]:
print("\n" + "="*60)
print("PHASE 2 COMPLETE!")
print("="*60)
print(f"\n✓ Final feature count: {X.shape[1]}")
print(f"✓ Original features: {X.shape[1] - 8} engineered features")


PHASE 2 COMPLETE!

✓ Final feature count: 20
✓ Original features: 12 engineered features
